In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display
import gradio as gr

In [ ]:
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1')
default_remote_models = ['gpt-4o-mini', 'gpt-5-nano', 'gpt-4.1-mini']

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
    remote_client = OpenAI()
else:
    print("OpenAI API Key not set. You can't use remote models.")

local_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
remote_client = OpenAI(api_key=openai_api_key)

In [ ]:
def get_local_models():
        try:
            response = local_client.models.list()
            return [{'name': model.id, 'type': 'local'} for model in response.data]
        except Exception as e:
            print(f'Error listing models: {e}')
            return []

In [ ]:
remote_models = [{'name': model.lower(), 'type': 'remote'}
                              for model in default_remote_models]

def get_models():
        local_models = get_local_models()
        models_dict = {i+1: model for i,
                       model in enumerate(remote_models + local_models)}
        return models_dict

get_models()

In [ ]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

In [ ]:
compile_command = ["g++", "-std=c++20", "-O3", "-march=native", "-flto", "-DNDEBUG", "main.cpp", "-o", "main"] 
run_command = ["./main"]

compile_test_command = ["g++", "-std=c++20", "-O3", "-march=native", "-flto", "-DNDEBUG", "tests.cpp", "-o", "tests"] 
run_test_command = ["./tests"]

In [ ]:
system_prompt_tests = '''
You are a helpful assistant and a testing expert.
You are given a C++ code and you need to generate unit tests for it using catch2 framework.
The tests should be in a separate file from the code.
The tests should be in the same directory as the code.
The tests should be named like the code file but with a .test.cpp extension.
The coverage should be 100 percent for the code.
The tests should be in the same language as the code.
The tests should be in the same style as the code.
The tests should be in the same format as the code.
The tests should be in the same structure as the code.
The tests should be in the same logic as the code.
The tests should be in the same meaning as the code.
Don't include any explanation or other text in the tests. respond only with the tests.
'''

system_prompt_convert = f'''
You are a C++ optimization expert.
Your task is to convert Python code into high performance C++ code.
If possible, use enhanced algorithms and data structures to make the code as fast as possible.
The code should be optimized for a system with this information: {system_info}
Use all the optimizations techniques you know to make the code as fast as possible.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
'''

In [ ]:
def convert_user_prompt(code):
    return f'''
    Convert the following Python code into high performance C++ code with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to be converted:
    {code}
    '''

def tests_user_prompt(code):
    return f'''
    Generate unit tests for the following C++ code using catch2 framework.
    {code}
    '''


In [ ]:
def write_output(cpp, output_file):
    cpp = cpp.replace('```cpp','').replace('```','')
    with open(output_file, "w") as f:
        f.write(cpp)
    
    print(f'Output written to {output_file}')

In [ ]:
def stream_and_write_model_response(prompt, model, system_prompt, output_file=None):

    if model.get('type') == 'remote':
        client: OpenAI = remote_client
    else:
        client: OpenAI = local_client

    try:
        print(f'Streaming response from model: {model['name']}')
        response = client.chat.completions.create(
            model=model['name'],
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': prompt}
            ],
            stream=True
        )
        result = ''
        for chunk in response:
            result += chunk.choices[0].delta.content or ''
            yield result

        write_output(result, output_file)
    except Exception as e:
        print(f'Error streaming model response: {e}')
        yield ''


In [ ]:
def stream_convert(prompt, model):
    yield from stream_and_write_model_response(convert_user_prompt(prompt), model, system_prompt_convert, "main.cpp")

def stream_tests(prompt, model):
    yield from stream_and_write_model_response(tests_user_prompt(prompt), model, system_prompt_tests, "tests.cpp")


In [ ]:
python_code = '''
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
'''

cpp_code = '''
#include <iostream>
#include <chrono>
#include <iomanip>
#include <immintrin.h>

// Using AVX2 vectorization to compute 8 iterations at once for speed.
// The calculation is:
// result = 1.0 - sum_{i=1}^N 1/(i*param1 - param2) + sum_{i=1}^N 1/(i*param1 + param2)
// Then multiplied by 4.
// We will unroll and vectorize for highest speed.

int main() {
    constexpr int64_t N = 200'000'000;
    constexpr double param1 = 4.0;
    constexpr double param2 = 1.0;

    // Using AVX2 double: 4 doubles in __m256d vector.
    // Process 4 iterations in one vector pass.
    // Then use 2 vectors to get 8 iterations per loop unroll for better ILP.

    auto t1 = std::chrono::high_resolution_clock::now();

    double result = 1.0;

    // We'll vectorize processing 8 iterations per loop:
    // Use two __m256d registers:
    // idx0 = i, i+1, i+2, i+3
    // idx1 = i+4, i+5, i+6, i+7

    __m256d v_param1 = _mm256_set1_pd(param1);
    __m256d v_param2 = _mm256_set1_pd(param2);

    double res0 = 0.0; // accumulator for negative terms sum(1/(i*param1 - param2))
    double res1 = 0.0; // accumulator for positive terms sum(1/(i*param1 + param2))

    int64_t i = 1;

    // Precompute constants for incrementing indices vector
    __m256d v_inc = _mm256_set_pd(1,1,1,1); // for increment

    for (; i <= N - 7; i += 8) {
        // Prepare index vectors for i..i+3 and i+4..i+7
        __m256d idx0 = _mm256_set_pd(double(i+3), double(i+2), double(i+1), double(i));
        __m256d idx1 = _mm256_set_pd(double(i+7), double(i+6), double(i+5), double(i+4));

        // Compute denominators: i*param1 - param2
        __m256d denom0a = _mm256_sub_pd(_mm256_mul_pd(idx0, v_param1), v_param2);
        __m256d denom1a = _mm256_sub_pd(_mm256_mul_pd(idx1, v_param1), v_param2);

        // Compute 1 / denom0a and denom1a
        __m256d inv0a = _mm256_div_pd(_mm256_set1_pd(1.0), denom0a);
        __m256d inv1a = _mm256_div_pd(_mm256_set1_pd(1.0), denom1a);

        // Compute denominators: i*param1 + param2
        __m256d denom0b = _mm256_add_pd(_mm256_mul_pd(idx0, v_param1), v_param2);
        __m256d denom1b = _mm256_add_pd(_mm256_mul_pd(idx1, v_param1), v_param2);

        // Compute 1 / denom0b and denom1b
        __m256d inv0b = _mm256_div_pd(_mm256_set1_pd(1.0), denom0b);
        __m256d inv1b = _mm256_div_pd(_mm256_set1_pd(1.0), denom1b);

        // Sum vectors
        // res0 += sum(inv0a + inv1a) [negative terms]
        // res1 += sum(inv0b + inv1b) [positive terms]

        // Horizontal adds for inv vectors
        __m256d a0 = _mm256_add_pd(inv0a, inv1a);
        __m256d b0 = _mm256_add_pd(inv0b, inv1b);

        // Reduce a0 and b0 horizontally
        // _mm256_hadd_pd adds pairwise horizontal elements but results still in vector form
        __m256d ha = _mm256_hadd_pd(a0, a0);
        __m256d hb = _mm256_hadd_pd(b0, b0);

        // Extract lower 2 doubles and sum
        double ha_lo = ((double*)&ha)[0] + ((double*)&ha)[2];
        double hb_lo = ((double*)&hb)[0] + ((double*)&hb)[2];

        res0 += ha_lo;
        res1 += hb_lo;
    }

    // Tail loop for remaining iterations
    for (; i <= N; i++) {
        double j1 = i * param1 - param2;
        double j2 = i * param1 + param2;
        res0 += 1.0 / j1;
        res1 += 1.0 / j2;
    }

    result = result - res0 + res1;
    result *= 4.0;

    auto t2 = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double> diff = t2 - t1;

    std::cout << std::fixed << std::setprecision(12);
    std::cout << "Result: " << result << "\n";
    std::cout << "Execution Time: " << diff.count() << " seconds\n";

    return 0;
}
'''

In [ ]:
models = get_models()
choices = [(model.get('name'), model) for model in models.values()]

with gr.Blocks(title="Week 4 — Python to C++ converter and C++ tests generator") as demo:
    gr.Markdown("# Python Toolbox")
    gr.Markdown("Convert Python code to C++ code and generate C++ tests for it.")

    with gr.Tab("Python to C++ converter"):
        with gr.Row():
            model_selector = gr.Dropdown(choices=choices, label='Select a model', value=choices[0][1])
        with gr.Row():
            gr.Markdown('Convert Python code to C++ code')
        with gr.Row():
            code_in = gr.Code(label="Python function", lines=18, language="python")
            code_out = gr.Code(label="C++ code", language="cpp")
        with gr.Row():
            code_btn = gr.Button("Convert to C++")
            code_btn.click(
                fn=stream_convert,
                inputs=[code_in, model_selector],
                outputs=code_out
            )
        with gr.Row():
            gr.Examples(
                examples=[python_code],
                inputs=code_in,
                label="Examples",
            )

    with gr.Tab("C++ tests generator"):
        with gr.Row():
            model_selector = gr.Dropdown(choices=choices, label='Select a model', value=choices[0][1])
        with gr.Row():
            gr.Markdown("### Unit test generator")
        with gr.Row():
            test_in = gr.Code(label="C++ code", lines=18, language="cpp")
            test_out = gr.Code(label="Generated tests (catch2)", language="cpp")
        with gr.Row():
            test_btn = gr.Button("Generate unit tests")
            test_btn.click(
                fn=stream_tests,
                inputs=[test_in, model_selector],
                outputs=test_out
            )
        with gr.Row():
            gr.Examples(
                examples=[cpp_code],
                inputs=test_in,
                label="Examples",
            )

demo.launch(inbrowser=True)

In [ ]:
import subprocess

def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")

In [ ]:
def compile_and_run_tests():
    try:
        subprocess.run(compile_test_command, check=True, text=True, capture_output=True)
        print(subprocess.run(run_test_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")


In [ ]:
compile_and_run()

In [ ]:
compile_and_run_tests()